# 06 — Standalone Hybrid Classification Assembly (Object-Refined Pixel Probabilities)

This notebook demonstrates **Classification Assembly** in Google Colab using **only the lean base install** of `lc-framework` — no Google Earth Engine login, no `luma-stack`, and no ML libraries required ([ADR-0005](../docs/adr/0005-standalone-modular-subpackages.md)).

It covers:
1. Installing `lc-framework` cleanly in Colab
2. Running **Object-Refined Pixel Probabilities** either with a `ProjectConfig` YAML or directly on plain GeoTIFF paths
3. Optional pure-pixel logit-space Bayesian smoothing (`bayesian_smooth_raster`)
4. Applying multi-resolution `MaskRule`s and Minimum Mapping Unit (MMU) filtering
5. Inspecting the refined `Class Probability Stack`, `Confusion Index` ($S/W$), stability map, and hard LULC map

In [ ]:
# 1. Colab Bootstrap — lean core install only (no GEE/ML conflicts)
# Replace <your-org> with your company GitHub organization
# !pip install -q "git+https://github.com/<your-org>/LC_Framework.git"

from pathlib import Path
import numpy as np
import rasterio

from lc_framework.config import ClassDefinition, ClassSchema, ProjectConfig
from lc_framework.assembly import (
    MaskRule,
    STABILITY_NODATA,
    bayesian_smooth_raster,
    run_hybrid_assembly,
)
from lc_framework.io import read_probability_stack

## 2. Mount Google Drive & Load Configuration (or Run Standalone)
If you have a `ProjectConfig` YAML on Drive, load it below. Otherwise, you can define a lightweight inline `ClassSchema` (or even pass `config=None` if your probability GeoTIFF already has a sidecar `*.manifest.json`).

In [ ]:
# Uncomment in Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

CONFIG_PATH = Path("../configs/example_project.yaml")
if CONFIG_PATH.exists():
    cfg = ProjectConfig.from_yaml(CONFIG_PATH)
    schema = cfg.class_schema
else:
    cfg = None
    schema = ClassSchema(
        classes=[
            ClassDefinition(id=1, name="Forest", color="#1b5e20"),
            ClassDefinition(id=2, name="Agroforestry", color="#66bb6a"),
            ClassDefinition(id=3, name="Cropland", color="#fbc02d"),
            ClassDefinition(id=4, name="Built-up", color="#e53935"),
            ClassDefinition(id=5, name="Waterbody", color="#0288d1"),
        ]
    )

# Paths to your GEE-exported (or locally generated) inputs
PROB_TIF = Path("data/pixel_probs_2025.tif")      # uint16 [0..10000] or float32 [0..1]
SEGMENT_TIF = Path("data/snic_segments_2025.tif") # int32 SNIC or SLIC Segment Label Raster
OUTPUT_DIR = Path("outputs/hybrid_2025")

print(f"Classes ({len(schema)}):")
for c in schema.classes:
    print(f"  {c.id:2d}: {c.name} ({c.color})")

## 3. Run Hybrid Classification Assembly
`run_hybrid_assembly` automatically:
- Reads the `Class Probability Stack` and its sidecar `*.manifest.json` (or works standalone without one)
- Aligns the `Segment Label Raster` and repairs GEE `int32` overflow (`2147483647`)
- Computes **Object-Refined Pixel Probabilities** with adaptive disagreement damping
- Writes a refined `Class Probability Stack` COG + manifest, `Confusion Index` (`NaN` nodata), stability map (`255` nodata), and hard LULC map (with optional mask rules & MMU)

In [ ]:
# Optional: define class-specific MMU thresholds (in pixels; 0 = preserve all patches)
mmu_table = {
    1: 20,  # Forest
    2: 10,  # Agroforestry
    3: 10,  # Cropland
    4: 0,   # Built-up (protected)
    5: 0,   # Waterbody (protected)
}

# Optional: multi-resolution auxiliary mask rules (any CRS/resolution via WarpedVRT)
mask_rules = [
    # MaskRule(
    #     from_classes=(5,),
    #     to_class=4,
    #     require_mask_path="data/settlement_mask.tif",
    #     require_value=1,
    #     name="Reassign false water inside confirmed settlements",
    # ),
]

if PROB_TIF.exists() and SEGMENT_TIF.exists():
    outputs = run_hybrid_assembly(
        prob_path=PROB_TIF,
        segment_path=SEGMENT_TIF,
        output_dir=OUTPUT_DIR,
        config=cfg,
        class_schema=schema if cfg is None else None,
        tau=0.65,
        adaptive_sensitivity=0.60,
        encoding="uint16",
        mask_rules=mask_rules or None,
        mmu_table=mmu_table,
    )
    print("Assembly complete:")
    print("  Refined Probabilities:", outputs.probabilities)
    print("  Hard LULC Map:        ", outputs.lulc)
    print("  Confusion Index:      ", outputs.confusion_index)
    print("  Stability Map:        ", outputs.stability)
elif PROB_TIF.exists():
    # No segment raster? You can still run standalone logit-space Bayesian smoothing:
    smoothed_path = bayesian_smooth_raster(
        PROB_TIF,
        OUTPUT_DIR / "pixel_probs_smoothed.tif",
        class_schema=schema,
        smoothness=1.5,
        window_size=7,
    )
    print("Bayesian smoothed probability stack:", smoothed_path)
else:
    print("Place your probability (and optional segment) GeoTIFFs in data/ and re-run.")

## 4. Quick QA Summary
Inspect the fraction of stable pixels ($CI \le \tau$) while excluding nodata (`STABILITY_NODATA = 255`).

In [ ]:
if PROB_TIF.exists() and SEGMENT_TIF.exists():
    refined = read_probability_stack(outputs.probabilities)
    with rasterio.open(outputs.stability) as src_s, rasterio.open(outputs.confusion_index) as src_ci:
        stab = src_s.read(1)
        ci = src_ci.read(1)
    valid = refined.valid_mask & (stab != STABILITY_NODATA)
    print(f"Valid pixels: {int(valid.sum()):,}")
    print(f"Mean Confusion Index: {float(np.nanmean(ci[valid])):.3f}")
    print(f"Stable pixel share (CI <= 0.65): {100.0 * float(stab[valid].mean()):.1f}%")